# GameGPT — Qwen3.5-9B 文本微调
### Unsloth + PAI-DSW 24GB | 两阶段分层 SFT

| 项目 | 配置 |
|---|---|
| 模型 | Qwen3.5-9B-Instruct（VLM 架构，只训练语言层）|
| 精度 | INT8 LoRA|
| LoRA | rank=32，alpha=64 |
| 等效 batch | 2 × 2 = 4 |
| Loss 策略 | Completion-only + improved 数据加权 |

> **说明**：Qwen3.5 是统一 Vision-Language 模型，无独立纯文本版本。  
> 通过 `finetune_vision_layers=False` 冻结视觉层，实现纯文本 SFT。  
> tokenizer 实际是 `Qwen3VLProcessor`，文本 encode 需通过 `.tokenizer` 子属性。


## 1. 环境清理与初始化
在开始之前，如果有正在运行的模型实例，则需要重启内核并释放显存，确保训练环境干净，避免显存溢出（OOM）。

In [ ]:
# import os
# os.kill(os.getpid(), 9)


: 

In [ ]:
import torch, gc
try:
    # 尝试删除显存中可能存在的变量（如模型、分词器、Trainer等）
    del model, tokenizer, text_tokenizer, trainer
except: pass

# 核心步骤：手工清空当前默认物理显卡的缓存，并触发垃圾回收机制
torch.cuda.empty_cache()
gc.collect()

print(f"当前由于之前的残留，仍然占用: {torch.cuda.memory_reserved()/1024**3:.1f} GB")

当前占用: 0.0 GB


In [ ]:
# 如果需要从 ModelScope 下载基础模型，可以在此处解开注释执行。
# 当前已将模型存在本地 ./模型/Qwen3.5-9B，因此跳过下载步骤
# ! pip install modelscope
# ! modelscope download --model Qwen/Qwen3.5-9B
# import numpy as np

## 环境安装（出于某种未知的原因，有时候环境会报错，只需要再次运行该单元格或者重启内核即可）

In [2]:
%%capture
import os, importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qqq", *args])

pip("--upgrade", "uv")

if importlib.util.find_spec("torch") is None:
    pip("torch", "triton>=3.3.0", "numpy", "pillow", "bitsandbytes",
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo",
        "unsloth[base] @ git+https://github.com/unslothai/unsloth")
elif importlib.util.find_spec("unsloth") is None:
    pip("unsloth")

pip("--upgrade", "--no-deps", "tokenizers", "trl", "unsloth", "unsloth_zoo")
pip("transformers")
print("✅ 安装完成")


In [ ]:
! pip install peft

In [ ]:
! pip show unsloth torch transformers

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  全局配置区 —— 所有超参集中在此处统一修改
# ══════════════════════════════════════════════════════════════════

# ── 模型 ──────────────────────────────────────────────────────────
MODEL_NAME     = "./模型/Qwen3.5-9B"       # 基础语言模型所在的相对目录路径
MAX_SEQ_LENGTH = 2048                      # 支持的最长序列Token数目，长于这的部分将被截断

# ── LoRA ──────────────────────────────────────────────────────────
LORA_R     = 32                            # LoRA矩阵的秩（通常在8-64之间，越高拟合越强）
LORA_ALPHA = 64                            # LoRA的Alpha系数，控制调整幅度

# ── 数据 ──────────────────────────────────────────────────────────
DATA_DIR        = "./数据/compressed_jsonl"   # 数据集所在目录，内部应含有*.jsonl文件
IMPROVED_WEIGHT = 2.5                # 如果启用了Weighted SFT，人工的高质量样本得到的loss衰减权重
STAGE           = 2                  # 标志当前属于哪一阶段：1为全量数据，2为仅高质量数据

# ── 训练 ──────────────────────────────────────────────────────────
EPOCHS           = 1                 # 整个数据集被遍历的总次数
PER_DEVICE_BATCH = 1                 # 为了防止OOM限制内存占用，设定每张GPU上只放1个Sample
GRAD_ACCUM       = 4                 # 梯度累加（将多次Batch合并后才执行步进更新），等效Batct=4
LEARNING_RATE    = 5e-5              # 学习率。此处为Stage 2的低学习率精调保护
WARMUP_RATIO     = 0.05              # Warm up比例控制开始时训练学习率渐渐上升以避免剧烈波动
SEED             = 42                # 随机数字种子

# ── 输出 ──────────────────────────────────────────────────────────
OUTPUT_DIR = "./模型/gamegpt_stage2"   # Adapter保存目录，避免覆盖 Stage1 的产出

In [ ]:
from unsloth import FastVisionModel
import torch

# ── 用户配置区 ─────────────────────────────────────────────────────────
# MODEL_NAME     = ".cache/modelscope/models/Qwen/Qwen3.5-9B"  # 本地路径 or HF repo
# ──────────────────────────────────────────────────────────────────────

# model, tokenizer = FastVisionModel.from_pretrained(
#     model_name     = MODEL_NAME,
#     max_seq_length = MAX_SEQ_LENGTH,
#     load_in_4bit   = False,   # bf16 LoRA，Qwen3.5 不推荐 4-bit
#     load_in_8bit     = True,   # ← INT8 量化
#     load_in_16bit  = False,
#     full_finetuning= False,
#     local_files_only = True,
#     device_map       = "cuda:0",  # 强制全部放 GPU，不 offload 到 CPU
# )
from peft import PeftModel

BASE_MODEL     = "./模型/Qwen3.5-9B"
STAGE1_CKPT    = "./模型/gamegpt_stage1/checkpoint-1000"  # ← 直接用checkpoint目录
STAGE2_CKPT    = "./模型/gamegpt_stage2/checkpoint-1270"
# Step 4：先加载 base model
model, tokenizer = FastVisionModel.from_pretrained(
    model_name       = BASE_MODEL,
    max_seq_length   = MAX_SEQ_LENGTH,
    load_in_8bit     = True,
    load_in_4bit     = False,
    full_finetuning  = False,
    local_files_only = True,
    device_map       = "cuda:0",
)

import torch, os
from safetensors.torch import load_file

# 先用 get_peft_model 初始化与 Stage1 完全相同的 LoRA 结构
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = SEED,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    max_seq_length = MAX_SEQ_LENGTH,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[unsloth.import_fixes|WARNING]Unsloth: Detected broken vLLM binary extension; disabling vLLM imports and continuing import.
Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


2026-04-19 11:49:21,032 - modelscope - WARNING - The passed in library_name,library_version,user_agent,force_download,proxiesetag_timeout,headers,endpoint will not be used in modelscope.
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.3.0. vLLM: 0.15.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 760/760 [03:15<00:00,  3.89it/s] 


## 2. 第一阶段成果继承与模型加载
在 Stage2 中，我们将使用高质量数据进行训练。首先加载 `Qwen3.5-9B` 的 Base 模型，然后使用 Stage1 产出的 checkpoint 进行 **权重注入**，随后继续微调。

In [ ]:
# 手动注入 checkpoint 权重
ckpt_file = os.path.join(STAGE1_CKPT, "adapter_model.safetensors")
if not os.path.exists(ckpt_file):
    ckpt_file = os.path.join(STAGE1_CKPT, "adapter_model.bin")

if ckpt_file.endswith(".safetensors"):
    state_dict = load_file(ckpt_file)
else:
    state_dict = torch.load(ckpt_file, map_location="cpu")

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"missing: {len(missing)}, unexpected: {len(unexpected)}")
print("✅ Stage 权重加载完成")
model.print_trainable_parameters()
# Qwen3.5 tokenizer 实际是 Qwen3VLProcessor，文本 encode 用子属性
text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
print(f"Processor 类型: {type(tokenizer).__name__}")
print(f"文本 Tokenizer: {type(text_tokenizer).__name__}")
# Step 4 末尾，model 加载完之后加这两行
model.config.max_position_embeddings = 262144
if hasattr(model.config, 'text_config'):
    model.config.text_config.max_position_embeddings = 262144
print(f"✅ 已修正 max_position_embeddings = {model.config.max_position_embeddings}")

missing: 1016, unexpected: 256
✅ Stage 权重加载完成
trainable params: 58,195,968 || all params: 9,468,009,712 || trainable%: 0.6147
trainable params: 58,195,968 || all params: 9,468,009,712 || trainable%: 0.6147
Processor 类型: Qwen3VLProcessor
文本 Tokenizer: TokenizersBackend
✅ 已修正 max_position_embeddings = 262144


In [ ]:
import json, random, math
from pathlib import Path
from collections import Counter
from datasets import Dataset

SYSTEM_MSG = (
    "你是三角洲行动游戏的战术 AI 助手，负责根据前20秒的对局上文，"
    "预测并描述主玩家在接下来5秒内的具体行为过程。"
    "用【主玩家...随后...最后...】的格式输出，不超过150字。"
)

def format_sample(sample):
    messages = [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": sample["input"]},
        {"role": "assistant", "content": sample["label"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

def load_jsonl(path):
    items = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                items.append(json.loads(s))
    return items

normal, improved = [], []
for fpath in sorted(Path(DATA_DIR).glob("*.jsonl")):
    items = load_jsonl(str(fpath))
    (improved if fpath.name.startswith("improved_") else normal).extend(items)

print(f"规则标注样本:  {len(normal):,}")
print(f"LLM高质量样本: {len(improved):,}")

records = []
for s in ([] if STAGE == 2 else normal):
    records.append({"text": format_sample(s), "__weight__": 1.0,
                    "label_type": s["meta"].get("label_type", "?")})
for s in improved:
    records.append({"text": format_sample(s), "__weight__": IMPROVED_WEIGHT,
                    "label_type": s["meta"].get("label_type", "?")})

random.seed(SEED)
random.shuffle(records)

# ── 先保留 label_type 列，用于过滤后统计 ──────────────────────────────
def tokenize_fn(examples):
    result = text_tokenizer(
        examples["text"],
        truncation         = False,
        padding            = False,
        add_special_tokens = False,
    )
    result["__weight__"]  = examples["__weight__"]
    result["label_type"]  = examples["label_type"]   # 暂时保留
    return result

raw_dataset   = Dataset.from_list(records)
train_dataset = raw_dataset.map(
    tokenize_fn,
    batched       = True,
    remove_columns= ["text"],                          # ← 只去掉 text，保留 label_type
    desc          = "Tokenizing",
)

# ── 过滤超长样本 ──────────────────────────────────────────────────────
before = len(train_dataset)
train_dataset = train_dataset.filter(
    lambda x: len(x["input_ids"]) <= MAX_SEQ_LENGTH,
    desc = "Filtering long samples",
)
after = len(train_dataset)
print(f"过滤超长样本: {before:,} → {after:,}  (丢弃 {before-after:,} 条，{(before-after)/before*100:.1f}%)\n")

# ── 按 label_type 统计过滤后各类样本数量 ─────────────────────────────
from collections import Counter
label_counts = Counter(train_dataset["label_type"])
print("过滤后各类样本数量:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:<20s}: {count:>7,} 条  ({count/after*100:.1f}%)")
print()

# ── 统计完毕后再删除 label_type 列 ───────────────────────────────────
train_dataset = train_dataset.remove_columns(["label_type"])

total_steps = math.ceil(len(train_dataset) / (PER_DEVICE_BATCH * GRAD_ACCUM)) * EPOCHS
print(f"总样本数: {len(train_dataset):,}  |  估算步数: {total_steps:,}")
print(f"列: {train_dataset.column_names}")

规则标注样本:  88,105
LLM高质量样本: 2,977


Filtering long samples: 100%|██████████| 2977/2977 [00:02<00:00, 1181.68 examples/s]

过滤超长样本: 2,977 → 2,540  (丢弃 437 条，14.7%)

过滤后各类样本数量:
  BeingResuce         :     482 条  (19.0%)
  Looting             :     467 条  (18.4%)
  Action              :     405 条  (15.9%)
  SkillStart          :     399 条  (15.7%)
  Grenade             :     394 条  (15.5%)
  Fire                :     393 条  (15.5%)

总样本数: 2,540  |  估算步数: 1,270
列: ['__weight__', 'input_ids', 'attention_mask']


In [8]:
# 数据已预 tokenize，input_ids 就是 token id 列表，直接 len() 统计
sample_ids = train_dataset[0]["input_ids"]
print("=" * 70)
print(f"第一条样本 token 数: {len(sample_ids)}")
print(f"解码预览:\n{text_tokenizer.decode(sample_ids[:100])}...")
print("=" * 70)

# 统计 token 长度分布
print("\n正在统计 token 长度分布（随机抽样 500 条）...")
import random as _rnd
samples_idx = _rnd.sample(range(len(train_dataset)), min(1000, len(train_dataset)))
lengths = sorted([len(train_dataset[i]["input_ids"]) for i in samples_idx])

print(f"  最短: {min(lengths)} tokens")
print(f"  最长: {max(lengths)} tokens")
print(f"  中位: {lengths[len(lengths)//2]} tokens")
print(f"  P90 : {lengths[int(len(lengths)*0.9)]} tokens")
print(f"  超过 MAX_SEQ_LENGTH({MAX_SEQ_LENGTH}) 的样本: "
      f"{sum(1 for l in lengths if l > MAX_SEQ_LENGTH)} / {len(lengths)}")


第一条样本 token 数: 283
解码预览:
<|im_start|>system
你是三角洲行动游戏的战术 AI 助手，负责根据前20秒的对局上文，预测并描述主玩家在接下来5秒内的具体行为过程。用【主玩家...随后...最后...】的格式输出，不超过150字。<|im_end|>
<|im_start|>user
【全局态势】
主玩家: 红狼(ID=6558, 队伍4) | 共17名干员 | 我方3人 / 敌方14...

正在统计 token 长度分布（随机抽样 500 条）...
  最短: 205 tokens
  最长: 2045 tokens
  中位: 936 tokens
  P90 : 1736 tokens
  超过 MAX_SEQ_LENGTH(2048) 的样本: 0 / 1000


## 3. DataCollator定义与SFTTrainer构建
如果是混合数据训练，并使用样本加权，必须自定义 `DataCollatorForCompletionOnlyLM`。由于 `trl` 内置的没法完美处理带 `__weight__` 特征的 Dataset列，这里自行重新封装 `WeightedSFTTrainer`。在其中只对 assistant 作出回答的目标文本进行损失函数的计算。

In [9]:
import torch
from trl import SFTTrainer, SFTConfig
from typing import Dict, Sequence

# ── Qwen3.5 ChatML 模板 token ────────────────────────────────────────
# Qwen3.5 的 ChatML 格式：<|im_start|>assistant\n ... <|im_end|>
# text_tokenizer 是从 Step 4 中得到的子属性，有 .encode() 方法
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
response_token_ids = text_tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f"Response template token ids: {response_token_ids}")
print(f"解码验证: {repr(text_tokenizer.decode(response_token_ids))}")


class DataCollatorForCompletionOnlyLM:
    """
    只对 assistant 回复部分计算 loss，prompt 部分 label 设为 -100。
    使用 text_tokenizer（Qwen3VLProcessor 的子 tokenizer）做 encode。
    """
    def __init__(self, response_token_ids: list, tokenizer):
        self.tokenizer = tokenizer
        self.response_token_ids = response_token_ids
        self.response_len = len(response_token_ids)

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        # 提取 __weight__（SFTTrainer 会把 dataset 的每列都传进来）
        weights = None
        if "__weight__" in instances[0]:
            weights = torch.tensor(
                [inst["__weight__"] for inst in instances], dtype=torch.float32
            )
            # 不传入模型，稍后在 compute_loss 里用
            instances = [{k: v for k, v in inst.items() if k != "__weight__"}
                         for inst in instances]

        # 标准 padding
        batch = self.tokenizer.pad(
            instances,
            padding=True,
            return_tensors="pt",
        )

        # 构建 labels，先全部复制 input_ids
        batch["labels"] = batch["input_ids"].clone()

        # 将 pad token 位置的 label 也 mask 掉
        pad_id = self.tokenizer.pad_token_id
        if pad_id is not None:
            batch["labels"][batch["labels"] == pad_id] = -100

        # 找 response_template，把 template 之前（含 template 自身）mask 为 -100
        for i in range(batch["input_ids"].shape[0]):
            input_ids = batch["input_ids"][i].tolist()
            found_idx = -1
            for idx in range(len(input_ids) - self.response_len + 1):
                if input_ids[idx: idx + self.response_len] == self.response_token_ids:
                    found_idx = idx
                    break
            if found_idx != -1:
                # mask 掉 prompt 部分（包括 template 本身）
                batch["labels"][i, : found_idx + self.response_len] = -100
            else:
                # 未找到模板则整条 mask（避免污染训练）
                batch["labels"][i, :] = -100

        if weights is not None:
            batch["__weight__"] = weights

        return batch

class WeightedSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        weights = inputs.pop("__weight__", None)
        result = super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
        loss = result[0] if return_outputs else result
        if weights is not None:
            loss = loss * weights.to(loss.device).float().mean()
        return (loss, result[1]) if return_outputs else loss

# 初始化 collator（用 text_tokenizer，有 .encode() 和 .pad()）
collator = DataCollatorForCompletionOnlyLM(
    response_token_ids = response_token_ids,
    tokenizer          = text_tokenizer,
)
print(f"\n✅ Completion-only Loss 启用")
print(f"   模板: {repr(RESPONSE_TEMPLATE)}")
print(f"   Token ids: {response_token_ids}（共 {len(response_token_ids)} 个）")

Response template token ids: [248045, 74455, 198]
解码验证: '<|im_start|>assistant\n'

✅ Completion-only Loss 启用
   模板: '<|im_start|>assistant\n'
   Token ids: [248045, 74455, 198]（共 3 个）


In [10]:
# @title 训练前显存统计
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory       = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}  |  显存上限: {max_memory} GB")
print(f"训练前已占用: {start_gpu_memory} GB")


GPU: NVIDIA A10  |  显存上限: 22.184 GB
训练前已占用: 16.188 GB


### stage2训练前检查stage1的权重是否注入成功

In [11]:
# 检查 LoRA 权重是否非零
import torch
for name, param in model.named_parameters():
    if "lora_A" in name and param.requires_grad:
        print(f"{name}: mean={param.data.abs().mean():.6f}, "
              f"std={param.data.std():.6f}")
        break  # 只看第一个就够了
# 验证 collator 的 mask 是否正确
batch = collator([train_dataset[i] for i in range(2)])
for i in range(2):
    labels = batch["labels"][i]
    valid = (labels != -100).sum().item()
    total = len(labels)
    print(f"样本{i}: 总token={total}, 有效label={valid} ({valid/total*100:.1f}%)")

base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_A.default.weight: mean=0.007807, std=0.009014
样本0: 总token=767, 有效label=88 (11.5%)
样本1: 总token=767, 有效label=97 (12.6%)


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

FastVisionModel.for_training(model)

trainer = WeightedSFTTrainer(
    model         = model,
    tokenizer     = text_tokenizer,
    train_dataset = train_dataset,
    data_collator = collator,
    args = SFTConfig(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        learning_rate               = LEARNING_RATE,
        warmup_steps                = 20,
        lr_scheduler_type           = "cosine",
        bf16                        = True,
        fp16                        = False,
        logging_steps               = 50,
        save_strategy               = "steps",   # ← 从 "epoch" 改为 "steps"
        save_steps                  = 250,       # ← 每 250 步保存一次 checkpoint
        save_total_limit            = 2,         # ← 最多保留 2 个 checkpoint（稍微多留一个）
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        max_grad_norm               = 1.0,
        seed                        = SEED,
        dataloader_num_workers      = 0,
        report_to                   = "none",
        remove_unused_columns       = False,
        max_seq_length              = MAX_SEQ_LENGTH,
        packing                     = False,
    ),
)
print("✅ Trainer 构建完成，开始训练...")
trainer_stats = trainer.train(resume_from_checkpoint=True)

✅ Trainer 构建完成，开始训练...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,540 | Num Epochs = 2 | Total steps = 1,270
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 58,195,968 of 9,468,009,712 (0.61% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1050,2.271257
1100,2.177100
1150,2.288743
1200,2.230583
1250,2.191063


In [14]:
# @title 训练后显存 & 耗时 
used_memory          = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"训练耗时 : {trainer_stats.metrics['train_runtime']:.1f}s  "
      f"({round(trainer_stats.metrics['train_runtime']/60, 1)} min)")
print(f"峰值显存 : {used_memory} GB "
      f"({round(used_memory/max_memory*100,1)}%)")
print(f"LoRA 增量: {used_memory_for_lora} GB "
      f"({round(used_memory_for_lora/max_memory*100,1)}%)")


训练耗时 : 3049.1s  (50.8 min)
峰值显存 : 18.824 GB (84.9%)
LoRA 增量: 2.636 GB (11.9%)


### 保存模型

三种保存方式：
1. 仅保存 LoRA adapter（体积最小，用于继续训练 / 合并）
2. 合并为完整 bf16 模型（用于 vLLM 部署）
3. 导出 GGUF（用于 llama.cpp / Ollama 本地部署）


### LoRA adapter 保存在模型lora_adapter

In [ ]:
ADAPTER_DIR = OUTPUT_DIR + "/lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ LoRA adapter 已保存: {ADAPTER_DIR}")


In [ ]:
# # 重新加载为 FP16（不量化）
# model_fp16, tokenizer = FastVisionModel.from_pretrained(
#     model_name       = BASE_MODEL,
#     max_seq_length   = MAX_SEQ_LENGTH,
#     load_in_8bit     = False,   # ← 关闭 INT8
#     load_in_4bit     = False,
#     dtype            = torch.bfloat16,
#     local_files_only = True,
#     device_map       = "cuda:0",
# )

# # 注入 LoRA 权重（用之前保存的 adapter）
# from safetensors.torch import load_file
# model_fp16 = FastVisionModel.get_peft_model(model_fp16, ...)  # 同训练配置
# state_dict = load_file(f"{LORA_DIR}/adapter_model.safetensors")
# model_fp16.load_state_dict(state_dict, strict=False)

# # 合并
# MERGED_DIR = OUTPUT_DIR + "/merged_bf16"
# model_fp16.save_pretrained_merged(
#     MERGED_DIR, tokenizer, save_method="merged_16bit"
# )
# print(f"✅ 合并完成: {MERGED_DIR}")

In [ ]:
# import json
# from pathlib import Path

# # ── 清理 config 中不可序列化的字段 ──────────────────────────────────
# def clean_config(model):
#     config = model.config
#     bad_keys = []
#     for k, v in vars(config).items():
#         if callable(v) and not isinstance(v, type):
#             bad_keys.append(k)
#     for k in bad_keys:
#         print(f"  移除不可序列化字段: {k}")
#         delattr(config, k)
#     # 同样清理 quantization_config（INT8 引入的）
#     if hasattr(config, "quantization_config"):
#         delattr(config, "quantization_config")
#         print("  移除 quantization_config")
#     return model

# model = clean_config(model)

# # ── 再尝试保存 GGUF ───────────────────────────────────────────────────
# GGUF_DIR = OUTPUT_DIR + "/gguf_q8"
# model.save_pretrained_gguf(
#     GGUF_DIR,
#     tokenizer,
#     quantization_method = "q8_0",
# )
# print(f"✅ GGUF 已保存: {GGUF_DIR}")

## 第二阶段精调（Stage 2，可选）

Stage 1 完成后，修改 **Step 3 全局配置区** 的以下参数，然后从 **Step 4** 重新运行：

```python
MODEL_NAME    = "./checkpoints/gamegpt_stage1/checkpoint"  # Stage1 adapter
OUTPUT_DIR    = "./checkpoints/gamegpt_stage2"
STAGE         = 2       # 只用 improved 3000 条精调
LEARNING_RATE = 5e-5    # 更小学习率防止过拟合
EPOCHS        = 1
```

**Stage 2 的作用**：在 Stage 1 建立的基础行为预测能力上，  
进一步用 LLM 高质量标注数据提升输出描述的自然语言质量。


# 推理阶段 (Inference)
在本阶段，我们将加载测试集并使用训练好的模型生成未来5秒的玩家战术行为描述。
包括构建针对推断过程所需的特殊Prompt指示和生成后的响应清理工作。

### 1. 系统提示词配置
利用构建良好的 `System Prompt`，我们可以为大模型注入游戏战术专家的“角色设定”，并明确规定它的输出格式必须是：`主玩家...随后...最后...` 的强序列描述。这样可以大幅度减少大模型的闲聊和多余解释。

In [ ]:
# 定义推理阶段使用的系统提示词（System Prompt）
# 该提示词对模型的输出格式和内容施加了严格约束，特别强调不使用推测性语气和时间戳，强制生成战术动作流描述。
SYSTEM_MSG = (
    "你是三角洲行动游戏的战术分析专家，擅长根据玩家的历史行为数据预测其接下来的行为。\n"
    "你会收到一段结构化的对局记录，包含：\n"
    "  - 【全局态势】：地图上的兵力分布与初始状态\n"
    "  - 【前段摘要】：0~15秒的移动方向、速度与关键动作\n"
    "  - 【关键窗口】：15~20秒的坐标、朝向、移动状态与事件\n\n"
    "请根据以上信息，预测主玩家在第20~25秒的行为过程，严格遵守要求：\n"
    "  1. 以【主玩家...随后...最后...】的格式描述行为链\n"
    "  2. 重点描述：移动方向与状态、开镜/关镜、关键动作（跳/趴/站/滑铲等）\n"
    "  3. 不超过150字，不要出现推测性语气（如'可能'、'也许'）\n"
    "  4. 禁止在行文中出现具体时间戳（如20.05s、21.3s等），只描述动作顺序和因果关系。如必须使用时间描述，则必须描述到24.50s到25.00s才能完成描述。\n"
    "  5. 结尾完成总结动作描述:\n"
    "     必须是人类可读的战术动作词汇，"
    "     例如：完成开火并击倒对手、丢雷后冲锋、释放技能、救援队友、搜刮物资、战术规避、协同作战等，"
    "     严禁直接输出英文和代码标签（Action、BeingResuce、Fire、Move等）。\n"
    "  6. 只输出描述内容，不要解释推理过程"
)

### 2. 推理前向函数设计与生成调优
在验证阶段，相比训练，我们要关闭反向传播梯度并加上 `FastVisionModel.for_inference(model)` 加速生成速度。同时在 `generate` 方法中增加诸如 `temperature=0.3` 这样的约束，让每次生成的行文更加保守、贴近战术事实，而不产生太大的随机发散。并且还要考虑到清洗部分模型夹带 `<think>` 标签返回的问题。

In [ ]:
import json, torch
from pathlib import Path
from tqdm import tqdm
import openpyxl
import re

# 开启推理模式，优化显存和计算速度（如禁用梯度）
FastVisionModel.for_inference(model)

# ── 决策类型映射 ──────────────────────────────────────────────────────
# 将数据集元数据 (meta) 里的英文 label_type 映射为模型易懂的中文指令描述
LABEL_META = {
    "Fire":        "开火",
    "SkillStart":  "放技能",
    "Grenade":     "丢雷",
    "Looting":     "搜索物资",
    "BeingResuce": "救援队友",
}

# 举例填充动作类型的描述，以指导模型生成丰富战术细节
ACTION_EXAMPLES = "换弹、开镜、左探头、右探头、回正探头、蹲、趴、站、跳、滑铲、关镜"


def parse_main_player(sample_input: str) -> str:
    """
    使用正则表达式，从 input 开头的固定格式中提取出真正的主玩家名称。
    样例解析格式：主玩家: 无名(ID=2982, 队伍3) -> 提取 "无名"
    """
    m = re.search(r"主玩家[：:]\s*(\S+?)\s*[\(（]", sample_input)
    return m.group(1) if m else "主玩家"


def build_hint(label_type: str, main_player: str) -> str:
    """
    依靠主玩家的用户名和标签类别，动态生成注入在 input 对话末尾的预测引导指令。
    """
    if label_type == "Action":
        action_desc = f"执行动作，如：{ACTION_EXAMPLES}"
    else:
        action_desc = LABEL_META.get(label_type, label_type)

    return (
        f"决策为{action_desc}，"
        f"预测描述接下来5s内主玩家{main_player}的行动，直接输出预测结果。"
    )


def build_prompt(sample: dict) -> str:
    """
    构建最终的推理输入 Prompt:
    sample 数据格式应为：
    {
        "input": "【全局态势】\n主玩家: 无名(ID=2982, 队伍3) ...",
        "meta":  {"label_type": "Action", "main_pid": "2982", ...}
    }
    """
    sample_input = sample.get("input", "")
    meta         = sample.get("meta", {})
    label_type   = meta.get("label_type", "")

    main_player  = parse_main_player(sample_input)
    hint         = build_hint(label_type, main_player)

    # 融合基础 input 数据与结尾的 Hint 引导词
    user_content = sample_input + "\n" + hint

    messages = [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": ""},
    ]
    # 利用模型的 chat template 生成对话流，并在最后强行加上大模型回复引导前缀
    prompt = text_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    # 此处切割处理是为了丢弃空文本，将 Prompt 停在 <|im_start|>assistant\n
    split_token = "<|im_start|>assistant\n"
    if split_token in prompt:
        prompt = prompt[:prompt.rfind(split_token) + len(split_token)]
    return prompt


def run_inference(sample_input: str) -> str:
    """
    单条样本的模型前向推理生成过程，使用 min_p 和 temperature 控制随机性。
    """
    prompt = build_prompt(sample_input)
    inputs = text_tokenizer(
        text=prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = 200,    # 输出最大限制 200 Token
            temperature    = 0.3,    # 温度0.3保证相对稳定的输出
            min_p          = 0.1,    # min_p 采样丢弃长尾概率词汇
            use_cache      = True,
            do_sample      = True,
        )

    # 截取新生成的 Token 切片（不属于 input_ids 的部分）
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    # 解码获取纯文本的推理结果
    return text_tokenizer.decode(new_ids, skip_special_tokens=True).strip()

### 3. 测试集载入与流式生成回写
最后，我们从测试集加载测试数据。然后根据是否附带验证 `label`（可能用于自测或者线下赛对比），去格式化地生成 `test_results.xlsx` 结果。使用此数据框架保存结果十分方便赛后进行归档。

In [ ]:
# ── 加载 test 集并开始清洗流程 ────────────────────────────────────────────
# 从制定路径读取压缩好的 JSONL 测试文件（不带 Label 或只用于验证）
TEST_FILE = "./数据/compressed_jsonl/test.jsonl"
SAVE_PATH = "./test_results.xlsx"   # 最后结果将统一推送到该 Excel 报表中

test_samples = []
with open(TEST_FILE, encoding="utf-8") as f:
    for line in f:
        s = line.strip()
        if s:
            test_samples.append(json.loads(s))

# 检测读入的数据集是否带有 label（如果有说明可能是用于指标验证）
has_label = "label" in test_samples[0] if test_samples else False
print(f"测试集样本数: {len(test_samples)}  |  含label: {has_label}")
import re

def clean_output(text: str) -> str:
    """
    模型推理后处理：
    针对可能残留在输出中的思维链标签 `<think>` 进行清洗剔除处理，
    消除模型在输出前生成分析链导致实际不符合要求的输出字数过长。
    """
    # 1. 去除 <think>...</think> 包裹内容（包含其中多行内容，点符任意匹配）
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # 2. 去除未配对的残留单一标签（如果模型中途被打断）
    text = re.sub(r"</?think>", "", text)
    # 3. 清理多余两层及以上的空行和首尾空白段
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

In [ ]:
# ── 批量推理保存逻辑 ──────────────────────────────────────────────────────────
# 开始预测所有记录并使用 tqdm 呈现等待进度条
predictions = []
for sample in tqdm(test_samples, desc="推理中"):
    raw_pred = run_inference(sample)         # 执行流式验证
    pred = clean_output(raw_pred)            # ← 将含有标签的文本传入后处理方法获取干净文本
    predictions.append(pred)

# ── 格式化导出到 xlsx（Excel）表格文件 ───────────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "推理结果"                 # 将Excel标签工作表改名

# 在表格中加入规范列名抬头
ws["A1"] = "题目序号"
ws["B1"] = "后5秒续写"
if has_label:
    # 增加额外的比对列
    ws["C1"] = "参考label"
    ws.column_dimensions["C"].width = 80  # 自动调整列宽以避免折叠看不清

ws.column_dimensions["A"].width = 12
ws.column_dimensions["B"].width = 80

# 遍历预测数据集列表，逐条追写进Excel单元格
for idx, (sample, pred) in enumerate(zip(test_samples, predictions), start=1):
    ws.cell(row=idx + 1, column=1, value=idx)               # A列写入索引数字
    ws.cell(row=idx + 1, column=2, value=pred)              # B列写入AI生成的干净推测描述
    if has_label:
        # 如果是评测文件，也将标准答案写入 C 列以供校验查错
        ws.cell(row=idx + 1, column=3, value=sample.get("label", ""))

# 保存文件落盘并打印成功状态
wb.save(SAVE_PATH)
print(f"✅ 结果已保存: {SAVE_PATH}  (共 {len(predictions)} 条)")

推理中: 100%|██████████| 110/110 [40:20<00:00, 22.00s/it]

✅ 结果已保存: ./test_results.xlsx  (共 110 条)
